# JupyterLite 教育ベンチマーク（総合版 v3）

大学初年次 Python 入門講義向けの**全ベンチマーク統合ノートブック**。
今回の改版で、フィードバックを反映して 3 つを追加しました。

1. **比較可能なスコア** — 端末×ブラウザ×回線をまたいで比べられる 0–100 の総合スコア（重み付き合成）。
2. **ネットワーク経路スコア** — 資産/パッケージ配信元への RTT・スループットを評価。
3. **WASM 層スコア** — Python を介さない純 WebAssembly の実行スループット（i32 / f64）と SIMD・threads 可否。

| 柱 | 内容 | 取得 |
|---|---|---|
| A 初回体験 | ページロード・DL量・端末・**ネットワーク経路** | △（真の cold load は外部ハーネス） |
| B 信頼性 | ストレージ容量・FS 入出力 | △（消失試験は手動） |
| C 実行時間 | import・計算・描画・**WASM 層** | ○ |
| ─ | **総合スコア** | ○ |

すべて Pyodide で完走するよう `try/except` 済み。最初のコードセルは `hello` を出力（ハーネス検出用）。
**使い方**: Run → Run All Cells。

In [ ]:
print("hello")

# Run All の起点付近で「ナビゲーション開始からの経過 ms」を記録（cold load + 初期化の概算プロキシ）
try:
    import js
    _PAGE_MS_AT_START = float(js.performance.now())
except Exception:
    _PAGE_MS_AT_START = None
_PAGE_MS_AT_START

## 0. 計測フックの準備

In [ ]:
import time

BENCH = []

class timer:
    """with timer("ラベル"): ... で囲んだ処理の実行時間を記録する。"""
    def __init__(self, label):
        self.label = label
    def __enter__(self):
        self.t0 = time.perf_counter()
        return self
    def __exit__(self, *exc):
        dt = time.perf_counter() - self.t0
        BENCH.append({"label": self.label, "seconds": round(dt, 4)})
        print(f"[bench] {self.label}: {dt:.4f}s")
        return False

## A. 初回ロード / 環境

`performance` と `navigator` から、端末情報・ページロード時間・概算ダウンロード量を取得。

> ダウンロード量は CDN（別オリジン）資産だと `transferSize` が 0 になり過小評価され得る。
> 真の cold load（DL + 初期化の壁時計）は外部 Playwright ハーネスで測ること。

In [ ]:
import sys

ENV = {
    "python_version": sys.version.split()[0],
    "platform": sys.platform,
    "is_pyodide": ("pyodide" in sys.modules) or (sys.platform == "emscripten"),
    "user_agent": None,
    "hardware_concurrency": None,
    "device_memory_gb": None,
}
try:
    import js
    ENV["user_agent"] = str(js.navigator.userAgent)
    try:
        ENV["hardware_concurrency"] = int(js.navigator.hardwareConcurrency)
    except Exception:
        pass
    try:
        ENV["device_memory_gb"] = float(js.navigator.deviceMemory)
    except Exception:
        pass
except Exception as e:
    print(f"[env] ブラウザ情報を取得できません: {e!r}")

ENV

In [ ]:
PAGE_LOAD = {
    "ms_since_navigation_at_first_cell": _PAGE_MS_AT_START,
    "dom_content_loaded_ms": None,
    "load_event_end_ms": None,
    "response_end_ms": None,
    "estimated_download_mb": None,
    "resource_count": None,
    "caveat": "別オリジンCDN資産は Timing-Allow-Origin が無いと transferSize=0 となり DL量を過小評価し得る",
}
try:
    import js
    navs = js.performance.getEntriesByType("navigation")
    if navs.length > 0:
        nav = navs[0]
        PAGE_LOAD["dom_content_loaded_ms"] = round(float(nav.domContentLoadedEventEnd), 1)
        PAGE_LOAD["load_event_end_ms"] = round(float(nav.loadEventEnd), 1)
        PAGE_LOAD["response_end_ms"] = round(float(nav.responseEnd), 1)
    total = 0.0
    n = 0
    for e in js.performance.getEntriesByType("resource"):
        try:
            total += float(e.transferSize)
        except Exception:
            pass
        n += 1
    PAGE_LOAD["estimated_download_mb"] = round(total / 1e6, 2)
    PAGE_LOAD["resource_count"] = n
except Exception as e:
    print(f"[page] ロードタイミングを取得できません: {e!r}")

PAGE_LOAD

## A-net. ネットワーク経路スコア

JupyterLite は client-side 実行なので「サーバ」= **資産/パッケージの配信元**。その経路を 3 通りで評価する。

- **passive**: 既にロード済み資産の Resource Timing から TTFB と実効スループットの中央値を算出。
- **active probe**: 同一オリジンを `no-store` で数回取得し、最小往復時間を粗く測る。
- **navigator.connection**: 実効回線種別・RTT・下り推定（Chromium 系のみ。Firefox/Safari は無し）。

> 別オリジン CDN は詳細が隠れることがある点に注意。

In [ ]:
NETWORK = {
    "connection_effective_type": None,
    "connection_rtt_ms": None,
    "connection_downlink_mbps": None,
    "probe_rtt_ms": None,
    "resource_ttfb_ms_median": None,
    "resource_throughput_mbps_median": None,
    "resource_count_with_size": None,
    "note": "client-side 実行。サーバ=資産/パッケージ配信元。別オリジンCDNは詳細が隠れる場合あり",
}

# navigator.connection（あれば）
try:
    import js
    conn = getattr(js.navigator, "connection", None)
    if conn is not None:
        try: NETWORK["connection_effective_type"] = str(conn.effectiveType)
        except Exception: pass
        try: NETWORK["connection_rtt_ms"] = float(conn.rtt)
        except Exception: pass
        try: NETWORK["connection_downlink_mbps"] = float(conn.downlink)
        except Exception: pass
except Exception as e:
    print(f"[net] connection 情報なし: {e!r}")

# passive: resource timing 解析
try:
    import js, statistics
    ttfbs, thrs = [], []
    for e in js.performance.getEntriesByType("resource"):
        try:
            ttfb = float(e.responseStart) - float(e.requestStart)
            if ttfb > 0:
                ttfbs.append(ttfb)
        except Exception:
            pass
        try:
            size = float(e.transferSize) or float(e.encodedBodySize)
            dur = float(e.responseEnd) - float(e.startTime)
            if size > 0 and dur > 0:
                thrs.append(size * 8 / (dur / 1000.0) / 1e6)  # Mbps
        except Exception:
            pass
    if ttfbs:
        NETWORK["resource_ttfb_ms_median"] = round(statistics.median(ttfbs), 1)
    if thrs:
        NETWORK["resource_throughput_mbps_median"] = round(statistics.median(thrs), 2)
        NETWORK["resource_count_with_size"] = len(thrs)
except Exception as e:
    print(f"[net] resource timing 解析失敗: {e!r}")

# active probe: 同一オリジンを no-store で 3 回、最小往復（粗いRTTプロキシ）
try:
    import js
    from pyodide.ffi import to_js
    base = str(js.location.href)
    sep = "&" if "?" in base else "?"
    best = None
    for k in range(3):
        t0 = js.performance.now()
        resp = await js.fetch(base + sep + "_probe=" + str(int(t0)) + "_" + str(k),
                              to_js({"cache": "no-store"}, dict_converter=js.Object.fromEntries))
        await resp.arrayBuffer()
        dt = js.performance.now() - t0
        best = dt if best is None else min(best, dt)
    if best is not None:
        NETWORK["probe_rtt_ms"] = round(best, 1)
except Exception as e:
    print(f"[net] アクティブプローブ失敗: {e!r}")

NETWORK

## B. ストレージ / 永続性プローブ

容量と仮想 FS の入出力を測る。

> タブ閉じ / 別端末 / キャッシュ消去 / プライベートモードでの消失は**ノート内では検証不可**。
> 手動シナリオ試験で確認（計画書の柱 B チェックリスト）。

In [ ]:
import time, os

STORAGE = {
    "quota_mb": None, "usage_mb": None, "persisted": None, "fs_roundtrip_s": None,
    "note": "タブ閉じ/別端末/キャッシュ消去による消失はノート内では検証不可（手動試験で確認）",
}
try:
    import js
    est = await js.navigator.storage.estimate()
    STORAGE["quota_mb"] = round(float(est.quota) / 1e6, 1)
    STORAGE["usage_mb"] = round(float(est.usage) / 1e6, 3)
except Exception as e:
    print(f"[storage] estimate 取得不可: {e!r}")
try:
    import js
    STORAGE["persisted"] = bool(await js.navigator.storage.persisted())
except Exception:
    pass
try:
    blob = "x" * 100_000
    t0 = time.perf_counter()
    with open("fs_test.txt", "w") as f:
        f.write(blob)
    with open("fs_test.txt") as f:
        back = f.read()
    STORAGE["fs_roundtrip_s"] = round(time.perf_counter() - t0, 5)
    assert back == blob
    os.remove("fs_test.txt")
except Exception as e:
    print(f"[storage] FS 往復失敗: {e!r}")

STORAGE

## C-1. Python 基礎（変数・制御構文・関数）

In [ ]:
with timer("python basics"):
    total = 0
    for i in range(1, 101):
        if i % 2 == 0:
            total += i

    def greet(name):
        return f"Hello, {name}!"

    result = {"evens_sum_1_to_100": total, "greeting": greet("students")}

result

## C-2. NumPy（配列と基本統計）

In [ ]:
with timer("import numpy"):
    import numpy as np

with timer("numpy ops"):
    a = np.arange(1_000_000, dtype=np.float64)
    stats = {"mean": float(a.mean()), "sum": float(a.sum()), "std": float(a.std())}

stats

## C-3. pandas（小さな CSV の読み込みと集計）

In [ ]:
csv_text = """name,group,score
Alice,A,82
Bob,B,75
Carol,A,90
Dave,B,68
Eve,A,88
Frank,B,79
"""

with open("sample.csv", "w") as f:
    f.write(csv_text)

with timer("import pandas"):
    import pandas as pd

with timer("read_csv + groupby"):
    df = pd.read_csv("sample.csv")
    summary = df.groupby("group")["score"].mean()

summary

## C-4. SymPy（記号計算・任意）

In [ ]:
try:
    with timer("import sympy"):
        import sympy as sp
    with timer("sympy factorint"):
        factors = sp.factorint(2**67 - 1)
    with timer("sympy expand"):
        x = sp.symbols("x")
        _ = sp.expand((x + 1) ** 50)
    print("[sympy] OK  factor(2^67 - 1) =", factors)
except Exception as e:
    print(f"[sympy] スキップ/失敗: {e!r}")

## C-5. matplotlib（可視化）

In [ ]:
with timer("import matplotlib"):
    import matplotlib.pyplot as plt

import numpy as np

with timer("line plot"):
    fig, ax = plt.subplots()
    ax.plot(range(10), [x**2 for x in range(10)], marker="o")
    ax.set_title("y = x^2")
    plt.show()

with timer("histogram"):
    rng = np.random.default_rng(0)
    fig, ax = plt.subplots()
    ax.hist(rng.normal(size=1000), bins=30)
    ax.set_title("normal sample (n=1000)")
    plt.show()

## C-6. WASM 層スコア（純 WebAssembly の素の速度）

Python インタプリタを介さず、ブラウザの WebAssembly エンジンで**整数 / 浮動小数の単純ループ**を直接実行し、
Mops/s（百万演算/秒）を測る。端末・ブラウザの素の実行性能を**機種横断で比較できる**指標。
あわせて SIMD 対応と threads 可否（SharedArrayBuffer + crossOriginIsolated）も判定する。

> 231 バイトの小さな WASM モジュールを埋め込んでいる（外部依存なし）。

In [ ]:
WASM = {
    "available": False,
    "i32_mops_per_s": None,
    "f64_mops_per_s": None,
    "simd_supported": None,
    "threads_capable": None,
    "note": "Python 層を介さない WebAssembly エンジンの素の実行速度",
}

WASM_B64 = "AGFzbQEAAAABCwJgAX8Bf2ABfwF8AwMCAAEHGQIJYmVuY2hfaTMyAAAJYmVuY2hfZjY0AAEKdQIxAQJ/QQEhAgJAA0AgASAATg0BIAJBjczlAGxB3+a74wNqIQIgAUEBaiEBDAALCyACC0ECAX8BfEQAAAAAAADwPyECAkADQCABIABODQEgAkTLGlDK///vP6JEmpmZmZmZuT+gIQIgAUEBaiEBDAALCyACCwA5BG5hbWUCGQIAAwABbgEBaQIDYWNjAQMAAW4BAWkCAXgDFwIAAgADYnJrAQJscAECAANicmsBAmxw"

try:
    import js, base64

    # threads 可否
    try:
        WASM["threads_capable"] = bool(getattr(js, "SharedArrayBuffer", None)) and bool(getattr(js, "crossOriginIsolated", False))
    except Exception:
        pass

    # SIMD 対応（wasm-feature-detect の検出モジュールで validate）
    try:
        simd_probe = bytes([0,97,115,109,1,0,0,0,1,5,1,96,0,1,123,3,2,1,0,10,10,1,8,0,65,0,253,15,253,98,11])
        sp_arr = js.Uint8Array.new(len(simd_probe)); sp_arr.assign(bytearray(simd_probe))
        WASM["simd_supported"] = bool(js.WebAssembly.validate(sp_arr))
    except Exception:
        pass

    # ベンチモジュールを読み込み・実行
    raw = base64.b64decode(WASM_B64)
    u8 = js.Uint8Array.new(len(raw)); u8.assign(bytearray(raw))
    res = await js.WebAssembly.instantiate(u8, js.Object.new())
    exports = res.instance.exports
    N = 50_000_000

    t0 = js.performance.now()
    exports.bench_i32(N)
    dt = (js.performance.now() - t0) / 1000.0
    if dt > 0:
        WASM["i32_mops_per_s"] = round(N / dt / 1e6, 1)

    t0 = js.performance.now()
    exports.bench_f64(N)
    dt = (js.performance.now() - t0) / 1000.0
    if dt > 0:
        WASM["f64_mops_per_s"] = round(N / dt / 1e6, 1)

    WASM["available"] = True
    print(f"[wasm] i32={WASM['i32_mops_per_s']} f64={WASM['f64_mops_per_s']} Mops/s, "
          f"simd={WASM['simd_supported']}, threads={WASM['threads_capable']}")
except Exception as e:
    print(f"[wasm] スキップ/失敗: {e!r}")

WASM

## C-7. ipywidgets（対話要素・任意）

In [ ]:
try:
    with timer("ipywidgets interact"):
        from ipywidgets import interact
        import numpy as np
        import matplotlib.pyplot as plt

        def plot_sine(freq=1.0):
            x = np.linspace(0, 2 * np.pi, 200)
            plt.figure()
            plt.plot(x, np.sin(freq * x))
            plt.title(f"sin({freq} x)")
            plt.show()

        interact(plot_sine, freq=(0.5, 5.0, 0.5))
    print("[ipywidgets] OK")
except Exception as e:
    print(f"[ipywidgets] スキップ/失敗: {e!r}")

## スコア化（比較可能な総合スコア）

各サブ指標を**目標値=100 とした 0–100 の相対スコア**に正規化し、教育用途向けの重みで合成する。
低いほど良い指標（時間・RTT）は `100 × 目標/実測`、高いほど良い指標（スループット・Mops/s）は `100 × 実測/目標`。

| サブスコア | 元指標 | 重み |
|---|---|---|
| load | 最初のセルまでの経過(ms) | 0.35 |
| network | RTT と スループット | 0.25 |
| pystack | pandas+matplotlib import + numpy ops 合計(s) | 0.20 |
| wasm | WASM i32 スループット(Mops/s) | 0.15 |
| storage | FS 往復(s) | 0.05 |

> **これは一つの便宜的な規約**であって絶対指標ではない。重み・目標値はクラスの実情に合わせて調整する。
> 取得できなかったサブスコアは除外し、残りの重みで再正規化する。総合スコアと各サブスコアの両方を出力する。

In [ ]:
def _lower_better(value, target):
    if value is None or value <= 0:
        return None
    return max(0.0, min(100.0, 100.0 * target / value))

def _higher_better(value, target):
    if value is None or value <= 0:
        return None
    return max(0.0, min(100.0, 100.0 * value / target))

def _bench_get(label):
    for m in BENCH:
        if m["label"] == label:
            return m["seconds"]
    return None

# 目標値（参照。クラスの実情に合わせて調整可）
TARGETS = {
    "load_ms": 8000,         # 最初のセル到達（ナビゲーション開始から、warm 想定）
    "rtt_ms": 100,           # ネットワーク往復
    "throughput_mbps": 10,   # 資産DLスループット
    "wasm_i32_mops": 200,    # 参照デバイスの WASM i32 スループット
    "pystack_s": 2.5,        # import pandas + matplotlib + numpy ops 合計
    "fs_s": 0.05,            # FS 往復
}

_pp, _pm, _no = _bench_get("import pandas"), _bench_get("import matplotlib"), _bench_get("numpy ops")
_vals = [v for v in (_pp, _pm, _no) if v is not None]
pystack_s = sum(_vals) if _vals else None

_rtt = NETWORK.get("probe_rtt_ms") or NETWORK.get("connection_rtt_ms")
_thr = NETWORK.get("resource_throughput_mbps_median") or NETWORK.get("connection_downlink_mbps")
_net_parts = [s for s in (_lower_better(_rtt, TARGETS["rtt_ms"]),
                          _higher_better(_thr, TARGETS["throughput_mbps"])) if s is not None]
network_sub = (sum(_net_parts) / len(_net_parts)) if _net_parts else None

SUBSCORES = {
    "load":    _lower_better(PAGE_LOAD.get("ms_since_navigation_at_first_cell"), TARGETS["load_ms"]),
    "network": network_sub,
    "pystack": _lower_better(pystack_s, TARGETS["pystack_s"]),
    "wasm":    _higher_better(WASM.get("i32_mops_per_s"), TARGETS["wasm_i32_mops"]),
    "storage": _lower_better(STORAGE.get("fs_roundtrip_s"), TARGETS["fs_s"]),
}
WEIGHTS = {"load": 0.35, "network": 0.25, "pystack": 0.20, "wasm": 0.15, "storage": 0.05}

_avail = {k: v for k, v in SUBSCORES.items() if v is not None}
_wsum = sum(WEIGHTS[k] for k in _avail)
composite = round(sum(SUBSCORES[k] * WEIGHTS[k] for k in _avail) / _wsum, 1) if _wsum > 0 else None

SCORE = {
    "composite_0_100": composite,
    "subscores": {k: (round(v, 1) if v is not None else None) for k, v in SUBSCORES.items()},
    "weights": WEIGHTS,
    "targets": TARGETS,
    "note": "教育用途向けの便宜的スコア。各サブスコアは目標値=100 の相対値(0-100にクリップ)。利用可能なサブスコアのみで再正規化。重み・目標は調整可。",
}

print("=== 総合スコア:", composite, "/ 100 ===")
for k, v in SCORE["subscores"].items():
    print(f"  {k:8}: {v}")
SCORE

## 結果の集計・出力

柱 A / B / C と総合スコアをまとめて JSON 表示し、`bench_results.json` に保存する。
端末・ブラウザ・回線ごとにこの 1 ファイルを回収すれば、`composite_0_100` でそのまま比較・順位付けできる。

In [ ]:
import json, time

payload = {
    "schema": "jupyterlite-edu-bench/v3",
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "score": SCORE,            # 比較可能な総合スコア
    "environment": ENV,        # 柱A: 端末・ブラウザ
    "page_load": PAGE_LOAD,    # 柱A: ロード時間・DL量
    "network": NETWORK,        # 柱A: ネットワーク経路
    "storage": STORAGE,        # 柱B: 容量・FS
    "wasm": WASM,              # 柱C: WASM 層
    "measurements": BENCH,     # 柱C: import/計算/描画
}

text = json.dumps(payload, ensure_ascii=False, indent=2)
print(text)

try:
    with open("bench_results.json", "w") as f:
        f.write(text)
    print()
    print("[saved] bench_results.json （ファイルブラウザからダウンロード可）")
except Exception as e:
    print()
    print(f"[warn] 保存失敗: {e!r}")